### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be
easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured
output.


In [3]:
import os
from langchain_groq import ChatGroq


model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

## Pydantic

In [4]:
from pydantic import BaseModel,Field

class Movies(BaseModel):
  title:str = Field(description="Title of the movie")
  year:int = Field(description="The year in which the movie was released")
  director:str = Field(description=" The director of the movie")
  rating : float = Field(description=" Rate the movie out of 10")
  

In [6]:
model_with_structure=model.with_structured_output(Movies, include_raw=True)
response = model_with_structure.invoke(" Tell about inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'eprrtrxqh', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movies'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 276, 'total_tokens': 308, 'completion_time': 0.056457838, 'completion_tokens_details': None, 'prompt_time': 0.05858586, 'prompt_tokens_details': None, 'queue_time': 0.181404106, 'total_time': 0.115043698}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d44cb-2c86-77f2-9d8b-62bb3109645c-0', tool_calls=[{'name': 'Movies', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': 'eprrtrxqh', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 276, 'output_tokens': 32

## Nested Structure

In [7]:
class Actor(BaseModel):
  name:str = Field(description=" Name of the actor")
  role:str   
  
class MovieDetails(BaseModel) :
  title:str
  year:int
  cast: list[Actor]
  genres: list[str]
  budget: float | None = Field(None, description="Budget in million USD")  
  

In [9]:
model_with_structure=model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke(" Tell about inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'nx4mgy15v', 'function': {'arguments': '{"budget":160,"cast":[{"name":"Leonardo DiCaprio","role":"Cobb"},{"name":"Joseph Gordon-Levitt","role":"Arthur"}],"genres":["Action","Sci-Fi"],"title":"Inception","year":2010}', 'name': 'MovieDetails'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 309, 'total_tokens': 379, 'completion_time': 0.1377279, 'completion_tokens_details': None, 'prompt_time': 0.031359969, 'prompt_tokens_details': None, 'queue_time': 0.05597513, 'total_time': 0.169087869}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d44cf-8fab-7c72-b7e5-344820af7184-0', tool_calls=[{'name': 'MovieDetails', 'args': {'budget': 160, 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Cobb'}, {'name': 'Joseph Gordon-Lev

### TypeDict

In [11]:
from typing import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The movie's rating out of 10"]

In [13]:
model_with_typedict = model.with_structured_output(MovieDict)
response= model_with_typedict.invoke("Tell about Avengers")
response

{'director': 'Joss Whedon',
 'rating': 8.1,
 'title': 'The Avengers',
 'year': 2012}

In [14]:
from typing import TypedDict, List, Optional

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: List[Actor]
    genres: List[str]
    budget: Optional[float]  # no Field here

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke(
    "Provide details about the movie Avengers"
)

print(response)


{'budget': 220000000, 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'}, {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'}], 'genres': ['Action', 'Adventure', 'Sci-Fi'], 'title': 'Avengers', 'year': 2012}


In [15]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True}

## Comparision Of All

In [21]:
# Pydantic

from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

# Load API key
load_dotenv()

# Model (Groq instead of gpt-5)
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

# Structured schema
class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

# Create agent with structured output
agent = create_agent(
    model=model,
    tools=[],  # no tools needed here
    response_format=ContactInfo
)

# Invoke agent
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

# Print result
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [17]:
# TypeDict

from typing import TypedDict
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

structured_model = model.with_structured_output(ContactInfo)

response = structured_model.invoke(
    "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
)

print(response)

{'email': 'john@example.com', 'name': 'John Doe', 'phone': '(555) 123-4567'}


In [18]:
# DataClass

from dataclasses import dataclass
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

structured_model = model.with_structured_output(ContactInfo)

response = structured_model.invoke(
    "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
)

print(response)

{'email': 'john@example.com', 'name': 'John Doe', 'phone': '(555) 123-4567'}
